# Stress Prediction v20 — New Pipeline

기존 파이프라인에서 벗어나 데이터 표현과 검증 구조를 다시 비교하는 실험용 노트북입니다.

## 단계

1. 최소 파생변수 생성
2. `OneHot` vs `Ordinal Encoding`
3. `KFold` vs target-bin `StratifiedKFold`
4. ExtraTrees multi-seed
5. Pair predictor `k=1 / 3 / 5`
6. Pair aggregation `nearest / mean / median / distance-weighted`
7. Hard override vs Soft override
8. 최종 제출 후보 생성

각 단계에서 가장 좋은 설정만 다음 단계로 전달합니다.


In [1]:
from __future__ import annotations

import os
import warnings
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)

TARGET = "stress_score"
ID_COLUMN = "ID"

RANDOM_SEED = 42
N_SPLITS = 5
SEEDS = [42, 2024, 777]
N_ESTIMATORS_PER_SEED = 600

TREE_QUANTILES = [0.535,0.540,0.545,0.550,0.555]
PAIR_WEIGHTS = [0.00,0.03,0.05,0.07,0.10,0.12,0.15]
PAIR_K_VALUES = [1, 3, 5]
PAIR_AGGREGATIONS = [
    "nearest",
    "mean",
    "median",
    "distance_weighted",
]

SOFT_OVERRIDE_ALPHAS = [
    0.0,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
]

OVERRIDE_THRESHOLDS = [
    None,
    0.10,
    0.12,
    0.14,
    0.16,
    0.18,
]

ROUND_OPTIONS = [2, None]

WINSORIZE_COLUMNS = [
    "mean_working",
    "diastolic_blood_pressure",
    "cholesterol",
    "glucose",
]

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
    "blood_pressure_ratio",
]

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FALLBACK = Path("/mnt/data/train(2).csv")
TEST_FALLBACK = Path("/mnt/data/test(2).csv")

print("SEEDS:", SEEDS)
print("Trees per seed:", N_ESTIMATORS_PER_SEED)


SEEDS: [42, 2024, 777]
Trees per seed: 600


## 1. 데이터 로드


In [ ]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"

if not train_path.exists() and TRAIN_FALLBACK.exists():
    train_path = TRAIN_FALLBACK

if not test_path.exists() and TEST_FALLBACK.exists():
    test_path = TEST_FALLBACK

train_raw = pd.read_csv(train_path)
test_raw = pd.read_csv(test_path)

y = train_raw[TARGET].reset_index(drop=True)

X_raw = (
    train_raw
    .drop(columns=[TARGET, ID_COLUMN])
    .reset_index(drop=True)
)

X_test_raw = (
    test_raw
    .drop(columns=[ID_COLUMN])
    .reset_index(drop=True)
)

numeric_raw_columns = X_raw.select_dtypes(
    include=["number"]
).columns.tolist()

print("Train shape:", train_raw.shape)
print("Test shape :", test_raw.shape)
display(train_raw.head())


## 2. 최소 피처 및 전처리


In [3]:
def safe_divide(
    numerator: pd.Series,
    denominator: pd.Series,
) -> pd.Series:
    return numerator / (denominator.abs() + 1e-6)


def compute_iqr_bounds(
    train_frame: pd.DataFrame,
    columns: list[str],
):
    bounds = {}

    for col in columns:
        q1, q3 = train_frame[col].quantile([0.25, 0.75])
        iqr = q3 - q1

        bounds[col] = (
            q1 - 1.5 * iqr,
            q3 + 1.5 * iqr,
        )

    return bounds


def apply_winsorize(
    frame: pd.DataFrame,
    bounds,
):
    result = frame.copy()

    for col, (lower, upper) in bounds.items():
        result[col] = result[col].clip(lower, upper)

    return result


def add_minimal_features(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()

    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    result["bmi"] = safe_divide(
        result["weight"],
        height_m.pow(2),
    )

    result["blood_pressure_ratio"] = safe_divide(
        sbp,
        dbp,
    )

    result["cholesterol_glucose_ratio"] = safe_divide(
        result["cholesterol"],
        result["glucose"],
    )

    result["missing_count"] = frame.isna().sum(axis=1)

    return result


def make_preprocessor(
    frame: pd.DataFrame,
    encoding: str,
) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    numerical = frame.select_dtypes(
        include=["number"]
    ).columns.tolist()

    if encoding == "onehot":
        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="missing",
                ),
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ])

    elif encoding == "ordinal":
        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="missing",
                ),
            ),
            (
                "encoder",
                OrdinalEncoder(
                    handle_unknown="use_encoded_value",
                    unknown_value=-1,
                    encoded_missing_value=-1,
                ),
            ),
        ])

    else:
        raise ValueError(f"알 수 없는 encoding: {encoding}")

    return ColumnTransformer([
        (
            "categorical",
            categorical_pipeline,
            categorical,
        ),
        (
            "numerical",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
            numerical,
        ),
    ])


## 3. CV split 생성


In [4]:
def make_cv_splits(
    X: pd.DataFrame,
    y: pd.Series,
    cv_type: str,
):
    if cv_type == "kfold":
        splitter = KFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=RANDOM_SEED,
        )
        return list(splitter.split(X))

    if cv_type == "stratified":
        # qcut 중복 경계를 허용해 안정적으로 bin 생성
        y_bins = pd.qcut(
            y,
            q=10,
            labels=False,
            duplicates="drop",
        )

        splitter = StratifiedKFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=RANDOM_SEED,
        )

        return list(
            splitter.split(
                X,
                y_bins,
            )
        )

    raise ValueError(f"알 수 없는 cv_type: {cv_type}")


## 4. ExtraTrees multi-seed 함수


In [5]:
# v23 parameter presets
TREE_CRITERION='squared_error'
MAX_FEATURES=0.95
MIN_SAMPLES_LEAF=1
TREE_QUANTILES=[0.532,0.533,0.534,0.535,0.536,0.537,0.538]
PAIR_WEIGHTS=[0.00,0.01,0.02,0.03,0.04,0.05]
print('v23 presets loaded')


v23 presets loaded


In [6]:
def fit_multiseed_tree_matrix(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
    encoding: str,
) -> np.ndarray:
    # 기존 실험에서 gender 제거가 유리했으므로 유지
    train_input = train_features.drop(
        columns=["gender"],
        errors="ignore",
    )

    val_input = val_features.drop(
        columns=["gender"],
        errors="ignore",
    )

    preprocessor = make_preprocessor(
        train_input,
        encoding=encoding,
    )

    train_matrix = preprocessor.fit_transform(
        train_input
    )

    val_matrix = preprocessor.transform(
        val_input
    )

    seed_matrices = []

    for seed in SEEDS:
        # v23: tune these globals before running search
        model = ExtraTreesRegressor(
            n_estimators=N_ESTIMATORS_PER_SEED,
            min_samples_leaf=MIN_SAMPLES_LEAF,
            max_features=MAX_FEATURES,
            criterion=TREE_CRITERION,
            random_state=seed,
            n_jobs=-1,
        )

        model.fit(
            train_matrix,
            target,
        )

        seed_matrix = np.column_stack([
            tree.predict(val_matrix)
            for tree in model.estimators_
        ])

        seed_matrices.append(seed_matrix)

    return np.concatenate(
        seed_matrices,
        axis=1,
    )


## 5. Encoding × CV 비교


In [7]:
encoding_cv_rows = []
encoding_cv_cache = {}

for encoding in ["onehot", "ordinal"]:
    for cv_type in ["kfold", "stratified"]:
        splits = make_cv_splits(
            X_raw,
            y,
            cv_type,
        )

        oof_pred = np.zeros(len(y))
        fold_tree_matrices = []

        for fold, (train_idx, val_idx) in enumerate(
            splits,
            start=1,
        ):
            train_fold_raw = (
                X_raw.iloc[train_idx]
                .reset_index(drop=True)
            )

            val_fold_raw = (
                X_raw.iloc[val_idx]
                .reset_index(drop=True)
            )

            y_train = (
                y.iloc[train_idx]
                .reset_index(drop=True)
            )

            bounds = compute_iqr_bounds(
                train_fold_raw,
                WINSORIZE_COLUMNS,
            )

            train_w = apply_winsorize(
                train_fold_raw,
                bounds,
            )

            val_w = apply_winsorize(
                val_fold_raw,
                bounds,
            )

            train_features = add_minimal_features(
                train_w
            )

            val_features = add_minimal_features(
                val_w
            )

            tree_matrix = fit_multiseed_tree_matrix(
                train_features,
                val_features,
                y_train,
                encoding=encoding,
            )

            tree_pred = np.quantile(
                tree_matrix,
                0.54,
                axis=1,
            )

            oof_pred[val_idx] = np.round(
                np.clip(tree_pred, 0, 1),
                2,
            )

            fold_tree_matrices.append({
                "val_idx": val_idx,
                "tree_matrix": tree_matrix,
            })

            print(
                f"{encoding}/{cv_type} "
                f"Fold {fold}/{N_SPLITS} 완료"
            )

        mae = mean_absolute_error(
            y,
            oof_pred,
        )

        key = (
            encoding,
            cv_type,
        )

        encoding_cv_rows.append({
            "encoding": encoding,
            "cv_type": cv_type,
            "tree_only_cv_mae": mae,
        })

        encoding_cv_cache[key] = {
            "splits": splits,
            "fold_tree_matrices": fold_tree_matrices,
            "oof_pred": oof_pred,
        }

encoding_cv_df = (
    pd.DataFrame(encoding_cv_rows)
    .sort_values("tree_only_cv_mae")
    .reset_index(drop=True)
)

display(encoding_cv_df)

BEST_ENCODING = str(
    encoding_cv_df.iloc[0]["encoding"]
)

BEST_CV_TYPE = str(
    encoding_cv_df.iloc[0]["cv_type"]
)

print("선택 Encoding:", BEST_ENCODING)
print("선택 CV:", BEST_CV_TYPE)


onehot/kfold Fold 1/5 완료
onehot/kfold Fold 2/5 완료
onehot/kfold Fold 3/5 완료
onehot/kfold Fold 4/5 완료
onehot/kfold Fold 5/5 완료
onehot/stratified Fold 1/5 완료
onehot/stratified Fold 2/5 완료
onehot/stratified Fold 3/5 완료
onehot/stratified Fold 4/5 완료
onehot/stratified Fold 5/5 완료
ordinal/kfold Fold 1/5 완료
ordinal/kfold Fold 2/5 완료
ordinal/kfold Fold 3/5 완료
ordinal/kfold Fold 4/5 완료
ordinal/kfold Fold 5/5 완료
ordinal/stratified Fold 1/5 완료
ordinal/stratified Fold 2/5 완료
ordinal/stratified Fold 3/5 완료
ordinal/stratified Fold 4/5 완료
ordinal/stratified Fold 5/5 완료


,encoding,cv_type,tree_only_cv_mae
0,ordinal,stratified,0.169043
1,onehot,stratified,0.169823
2,onehot,kfold,0.173873
3,ordinal,kfold,0.175850


선택 Encoding: ordinal
선택 CV: stratified


## 6. Pair predictor k / aggregation 함수


In [8]:
def empirical_rank_transform(
    train_values: np.ndarray,
    val_values: np.ndarray,
):
    median = np.nanmedian(train_values)

    train_values = np.nan_to_num(
        train_values,
        nan=median,
    )

    val_values = np.nan_to_num(
        val_values,
        nan=median,
    )

    sorted_train = np.sort(train_values)

    train_rank = (
        np.searchsorted(
            sorted_train,
            train_values,
            side="right",
        )
        / len(train_values)
    )

    val_rank = (
        np.searchsorted(
            sorted_train,
            val_values,
            side="right",
        )
        / len(train_values)
    )

    return (
        train_rank.astype(np.float32),
        val_rank.astype(np.float32),
    )


def aggregate_neighbor_targets(
    targets: np.ndarray,
    distances: np.ndarray,
    aggregation: str,
) -> np.ndarray:
    if aggregation == "nearest":
        return targets[:, 0]

    if aggregation == "mean":
        return targets.mean(axis=1)

    if aggregation == "median":
        return np.median(targets, axis=1)

    if aggregation == "distance_weighted":
        weights = 1.0 / (distances + 1e-6)
        weights = weights / weights.sum(
            axis=1,
            keepdims=True,
        )

        return (
            targets * weights
        ).sum(axis=1)

    raise ValueError(
        f"알 수 없는 aggregation: {aggregation}"
    )


def pair_prediction_general(
    train_features: pd.DataFrame,
    val_features: pd.DataFrame,
    target: pd.Series,
    k: int,
    aggregation: str,
) -> np.ndarray:
    train_rank = {}
    val_rank = {}

    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = (
            empirical_rank_transform(
                train_features[column]
                .to_numpy(dtype=float),
                val_features[column]
                .to_numpy(dtype=float),
            )
        )

    target_values = target.to_numpy(dtype=float)
    pair_predictions = []

    for first, second in combinations(
        PAIR_COLUMNS,
        2,
    ):
        distance = (
            np.abs(
                val_rank[first][:, None]
                - train_rank[first][None, :]
            )
            + np.abs(
                val_rank[second][:, None]
                - train_rank[second][None, :]
            )
        )

        nearest_idx = np.argpartition(
            distance,
            kth=k - 1,
            axis=1,
        )[:, :k]

        nearest_dist = np.take_along_axis(
            distance,
            nearest_idx,
            axis=1,
        )

        nearest_targets = target_values[
            nearest_idx
        ]

        pair_pred = aggregate_neighbor_targets(
            nearest_targets,
            nearest_dist,
            aggregation,
        )

        pair_predictions.append(pair_pred)

    return np.quantile(
        np.column_stack(pair_predictions),
        0.48,
        axis=1,
    )


## 7. 선택된 Encoding/CV에서 Pair 비교


In [9]:
best_splits = make_cv_splits(
    X_raw,
    y,
    BEST_CV_TYPE,
)

pair_rows = []
pair_fold_cache = []

for fold, (train_idx, val_idx) in enumerate(
    best_splits,
    start=1,
):
    train_fold_raw = (
        X_raw.iloc[train_idx]
        .reset_index(drop=True)
    )

    val_fold_raw = (
        X_raw.iloc[val_idx]
        .reset_index(drop=True)
    )

    y_train = (
        y.iloc[train_idx]
        .reset_index(drop=True)
    )

    bounds = compute_iqr_bounds(
        train_fold_raw,
        WINSORIZE_COLUMNS,
    )

    train_w = apply_winsorize(
        train_fold_raw,
        bounds,
    )

    val_w = apply_winsorize(
        val_fold_raw,
        bounds,
    )

    train_features = add_minimal_features(
        train_w
    )

    val_features = add_minimal_features(
        val_w
    )

    tree_matrix = fit_multiseed_tree_matrix(
        train_features,
        val_features,
        y_train,
        encoding=BEST_ENCODING,
    )

    pair_predictions = {}

    for k in PAIR_K_VALUES:
        for aggregation in PAIR_AGGREGATIONS:
            key = (
                k,
                aggregation,
            )

            pair_predictions[key] = (
                pair_prediction_general(
                    train_features,
                    val_features,
                    y_train,
                    k=k,
                    aggregation=aggregation,
                )
            )

    pair_fold_cache.append({
        "val_idx": val_idx,
        "train_idx": train_idx,
        "tree_matrix": tree_matrix,
        "pair_predictions": pair_predictions,
        "train_raw": train_fold_raw,
        "val_raw": val_fold_raw,
        "y_train": y_train,
    })

    print(
        f"Pair cache Fold {fold}/{N_SPLITS} 완료"
    )

for k in PAIR_K_VALUES:
    for aggregation in PAIR_AGGREGATIONS:
        oof_pred = np.zeros(len(y))

        for cache in pair_fold_cache:
            tree_pred = np.quantile(
                cache["tree_matrix"],
                0.54,
                axis=1,
            )

            pair_pred = cache["pair_predictions"][
                (k, aggregation)
            ]

            blended = (
                0.70 * tree_pred
                + 0.30 * pair_pred
            )

            oof_pred[
                cache["val_idx"]
            ] = np.round(
                np.clip(blended, 0, 1),
                2,
            )

        mae = mean_absolute_error(
            y,
            oof_pred,
        )

        pair_rows.append({
            "k": k,
            "aggregation": aggregation,
            "cv_mae": mae,
        })

pair_df = (
    pd.DataFrame(pair_rows)
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

display(pair_df)

BEST_PAIR_K = int(
    pair_df.iloc[0]["k"]
)

BEST_PAIR_AGGREGATION = str(
    pair_df.iloc[0]["aggregation"]
)

print("선택 Pair k:", BEST_PAIR_K)
print(
    "선택 Pair aggregation:",
    BEST_PAIR_AGGREGATION,
)


Pair cache Fold 1/5 완료
Pair cache Fold 2/5 완료
Pair cache Fold 3/5 완료
Pair cache Fold 4/5 완료
Pair cache Fold 5/5 완료


,k,aggregation,cv_mae
0,1,nearest,0.159063
1,1,mean,0.159063
2,1,median,0.159063
3,1,distance_weighted,0.159063
4,3,nearest,0.159063
5,3,distance_weighted,0.166060
6,5,nearest,0.167520
7,5,distance_weighted,0.168637
8,3,median,0.177447
9,5,median,0.181623


선택 Pair k: 1
선택 Pair aggregation: nearest


## 8. Soft / Hard override 비교


In [10]:
def nearest_neighbor_lookup(
    train_raw_frame: pd.DataFrame,
    val_raw_frame: pd.DataFrame,
    target: pd.Series,
):
    train_numeric = train_raw_frame[
        numeric_raw_columns
    ].copy()

    val_numeric = val_raw_frame[
        numeric_raw_columns
    ].copy()

    medians = train_numeric.median()

    train_numeric = train_numeric.fillna(
        medians
    )

    val_numeric = val_numeric.fillna(
        medians
    )

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(
        train_numeric
    )

    val_scaled = scaler.transform(
        val_numeric
    )

    tree = cKDTree(train_scaled)

    nearest_dist, nearest_idx = tree.query(
        val_scaled,
        k=1,
    )

    nearest_target = (
        target.to_numpy(dtype=float)[nearest_idx]
    )

    return nearest_dist, nearest_target


for cache in pair_fold_cache:
    nearest_dist, nearest_target = nearest_neighbor_lookup(
        cache["train_raw"],
        cache["val_raw"],
        cache["y_train"],
    )

    cache["nearest_dist"] = nearest_dist
    cache["nearest_target"] = nearest_target


def apply_soft_override(
    prediction: np.ndarray,
    nearest_dist: np.ndarray,
    nearest_target: np.ndarray,
    threshold: float | None,
    alpha: float,
) -> np.ndarray:
    result = prediction.copy()

    if threshold is None or alpha == 0.0:
        return result

    mask = nearest_dist < threshold

    result[mask] = (
        (1.0 - alpha) * result[mask]
        + alpha * nearest_target[mask]
    )

    return result


## 9. 최종 Grid Search


In [11]:
search_rows = []

for tree_q in TREE_QUANTILES:
    for pair_weight in PAIR_WEIGHTS:
        for threshold in OVERRIDE_THRESHOLDS:
            for alpha in SOFT_OVERRIDE_ALPHAS:
                for decimals in ROUND_OPTIONS:
                    oof_pred = np.zeros(
                        len(y),
                        dtype=float,
                    )

                    for cache in pair_fold_cache:
                        tree_pred = np.quantile(
                            cache["tree_matrix"],
                            tree_q,
                            axis=1,
                        )

                        pair_pred = cache[
                            "pair_predictions"
                        ][
                            (
                                BEST_PAIR_K,
                                BEST_PAIR_AGGREGATION,
                            )
                        ]

                        diff = np.abs(tree_pred - pair_pred)

                        blended = tree_pred.copy()
                        mask = diff < 0.03
                        blended[mask] = (
                            (1.0 - pair_weight) * tree_pred[mask]
                            + pair_weight * pair_pred[mask]
                        )

                        adjusted = apply_soft_override(
                            blended,
                            cache["nearest_dist"],
                            cache["nearest_target"],
                            threshold=threshold,
                            alpha=alpha,
                        )

                        adjusted = np.clip(
                            adjusted,
                            0.0,
                            1.0,
                        )

                        if decimals is not None:
                            adjusted = np.round(
                                adjusted,
                                decimals,
                            )

                        oof_pred[
                            cache["val_idx"]
                        ] = adjusted

                    mae = mean_absolute_error(
                        y,
                        oof_pred,
                    )

                    search_rows.append({
                        "cv_mae": mae,
                        "encoding": BEST_ENCODING,
                        "cv_type": BEST_CV_TYPE,
                        "tree_quantile": tree_q,
                        "pair_k": BEST_PAIR_K,
                        "pair_aggregation": (
                            BEST_PAIR_AGGREGATION
                        ),
                        "pair_weight": pair_weight,
                        "override_threshold": threshold,
                        "override_alpha": alpha,
                        "round_decimals": decimals,
                    })

search_df = (
    pd.DataFrame(search_rows)
    .sort_values("cv_mae")
    .reset_index(drop=True)
)

display(search_df.head(40))

print(
    "v20 최적 OOF MAE:",
    float(search_df.iloc[0]["cv_mae"]),
)


,cv_mae,encoding,cv_type,tree_quantile,pair_k,pair_aggregation,pair_weight,override_threshold,override_alpha,round_decimals
0,0.168625,ordinal,stratified,0.532,1,nearest,0.05,0.14,0.3,NaN
1,0.168625,ordinal,stratified,0.532,1,nearest,0.05,0.12,0.3,NaN
2,0.168625,ordinal,stratified,0.532,1,nearest,0.05,0.16,0.3,NaN
3,0.168625,ordinal,stratified,0.532,1,nearest,0.05,0.18,0.3,NaN
4,0.168628,ordinal,stratified,0.532,1,nearest,0.04,0.16,0.3,NaN
5,0.168628,ordinal,stratified,0.532,1,nearest,0.04,0.12,0.3,NaN
6,0.168628,ordinal,stratified,0.532,1,nearest,0.04,0.18,0.3,NaN
7,0.168628,ordinal,stratified,0.532,1,nearest,0.04,0.14,0.3,NaN
8,0.168631,ordinal,stratified,0.532,1,nearest,0.03,0.12,0.3,NaN
9,0.168631,ordinal,stratified,0.532,1,nearest,0.03,0.18,0.3,NaN


v20 최적 OOF MAE: 0.16862548066666666


## 10. 최종 제출 후보


In [12]:
def row_to_config(
    row: pd.Series,
) -> dict:
    return {
        "tree_quantile": float(
            row["tree_quantile"]
        ),
        "pair_weight": float(
            row["pair_weight"]
        ),
        "override_threshold": (
            None
            if pd.isna(
                row["override_threshold"]
            )
            else float(
                row["override_threshold"]
            )
        ),
        "override_alpha": float(
            row["override_alpha"]
        ),
        "round_decimals": (
            None
            if pd.isna(
                row["round_decimals"]
            )
            else int(
                row["round_decimals"]
            )
        ),
        "cv_mae": float(
            row["cv_mae"]
        ),
    }


candidate_rows = [
    ("cv_best", search_df.iloc[0]),
]

soft_rows = search_df[
    search_df["override_alpha"].between(
        0.2,
        0.4,
    )
]

if len(soft_rows):
    candidate_rows.append(
        ("soft_override", soft_rows.iloc[0])
    )

hard_rows = search_df[
    search_df["override_alpha"] == 1.0
]

if len(hard_rows):
    candidate_rows.append(
        ("hard_override", hard_rows.iloc[0])
    )

no_override_rows = search_df[
    search_df["override_alpha"] == 0.0
]

if len(no_override_rows):
    candidate_rows.append(
        ("no_override", no_override_rows.iloc[0])
    )

tree_heavy_rows = search_df[
    search_df["pair_weight"]
    == min(PAIR_WEIGHTS)
]

if len(tree_heavy_rows):
    candidate_rows.append(
        ("tree_heavy", tree_heavy_rows.iloc[0])
    )

FINAL_CONFIGS = []
seen = set()

for name, row in candidate_rows:
    config = row_to_config(row)

    key = (
        config["tree_quantile"],
        config["pair_weight"],
        config["override_threshold"],
        config["override_alpha"],
        config["round_decimals"],
    )

    if key not in seen:
        seen.add(key)
        FINAL_CONFIGS.append(
            (name, config)
        )

FINAL_CONFIGS = FINAL_CONFIGS[:5]

for name, config in FINAL_CONFIGS:
    print(name, config)


cv_best {'tree_quantile': 0.532, 'pair_weight': 0.05, 'override_threshold': 0.14, 'override_alpha': 0.3, 'round_decimals': None, 'cv_mae': 0.16862548066666666}
no_override {'tree_quantile': 0.532, 'pair_weight': 0.05, 'override_threshold': 0.12, 'override_alpha': 0.0, 'round_decimals': None, 'cv_mae': 0.16875448066666668}
tree_heavy {'tree_quantile': 0.532, 'pair_weight': 0.0, 'override_threshold': 0.12, 'override_alpha': 0.3, 'round_decimals': None, 'cv_mae': 0.16863948}


## 11. 전체 데이터 학습 및 제출 파일 생성


In [13]:
if not sample_path.exists():
    submission_template = pd.DataFrame({
        ID_COLUMN: test_raw[ID_COLUMN],
        TARGET: 0.0,
    })
else:
    submission_template = pd.read_csv(
        sample_path
    )

bounds = compute_iqr_bounds(
    X_raw,
    WINSORIZE_COLUMNS,
)

train_w = apply_winsorize(
    X_raw,
    bounds,
)

test_w = apply_winsorize(
    X_test_raw,
    bounds,
)

train_features = add_minimal_features(
    train_w
)

test_features = add_minimal_features(
    test_w
)

test_tree_matrix = fit_multiseed_tree_matrix(
    train_features,
    test_features,
    y,
    encoding=BEST_ENCODING,
)

test_pair_pred = pair_prediction_general(
    train_features,
    test_features,
    y,
    k=BEST_PAIR_K,
    aggregation=BEST_PAIR_AGGREGATION,
)

test_nearest_dist, test_nearest_target = (
    nearest_neighbor_lookup(
        X_raw,
        X_test_raw,
        y,
    )
)

saved_paths = []

for rank, (name, config) in enumerate(
    FINAL_CONFIGS,
    start=1,
):
    tree_pred = np.quantile(
        test_tree_matrix,
        config["tree_quantile"],
        axis=1,
    )

    diff = np.abs(tree_pred - test_pair_pred)
    blended = tree_pred.copy()
    mask = diff < 0.03
    blended[mask] = (
        (1.0-config["pair_weight"])*tree_pred[mask]
        + config["pair_weight"]*test_pair_pred[mask]
    )

    adjusted = apply_soft_override(
        blended,
        test_nearest_dist,
        test_nearest_target,
        threshold=config[
            "override_threshold"
        ],
        alpha=config[
            "override_alpha"
        ],
    )

    adjusted = np.clip(
        adjusted,
        0.0,
        1.0,
    )

    if config["round_decimals"] is not None:
        adjusted = np.round(
            adjusted,
            config["round_decimals"],
        )

    submission = submission_template.copy()
    submission[TARGET] = adjusted

    filename = (
        f"submit_v20_rank{rank}_{name}"
        f"_{BEST_ENCODING}_{BEST_CV_TYPE}"
        f"_k{BEST_PAIR_K}"
        f"_{BEST_PAIR_AGGREGATION}"
        f"_tq{config['tree_quantile']:.3f}"
        f"_w{config['pair_weight']:.2f}"
        f"_a{config['override_alpha']:.1f}.csv"
    )

    output_file = OUTPUT_DIR / filename

    submission.to_csv(
        output_file,
        index=False,
    )

    saved_paths.append(output_file)

    print("\nSaved:", output_file)
    print("Config:", config)
    display(submission.head())

print("\n생성 완료")
for path in saved_paths:
    print(path)



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_v20_rank1_cv_best_ordinal_stratified_k1_nearest_tq0.532_w0.05_a0.3.csv
Config: {'tree_quantile': 0.532, 'pair_weight': 0.05, 'override_threshold': 0.14, 'override_alpha': 0.3, 'round_decimals': None, 'cv_mae': 0.16862548066666666}


,ID,stress_score
0,TEST_0000,0.64
1,TEST_0001,0.97
2,TEST_0002,0.50
3,TEST_0003,0.60
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_v20_rank2_no_override_ordinal_stratified_k1_nearest_tq0.532_w0.05_a0.0.csv
Config: {'tree_quantile': 0.532, 'pair_weight': 0.05, 'override_threshold': 0.12, 'override_alpha': 0.0, 'round_decimals': None, 'cv_mae': 0.16875448066666668}


,ID,stress_score
0,TEST_0000,0.64
1,TEST_0001,0.97
2,TEST_0002,0.50
3,TEST_0003,0.60
4,TEST_0004,0.53



Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_v20_rank3_tree_heavy_ordinal_stratified_k1_nearest_tq0.532_w0.00_a0.3.csv
Config: {'tree_quantile': 0.532, 'pair_weight': 0.0, 'override_threshold': 0.12, 'override_alpha': 0.3, 'round_decimals': None, 'cv_mae': 0.16863948}


,ID,stress_score
0,TEST_0000,0.64
1,TEST_0001,0.97
2,TEST_0002,0.50
3,TEST_0003,0.60
4,TEST_0004,0.53



생성 완료
/Users/bitsaem/Desktop/stress_project/8_6/submit_v20_rank1_cv_best_ordinal_stratified_k1_nearest_tq0.532_w0.05_a0.3.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_v20_rank2_no_override_ordinal_stratified_k1_nearest_tq0.532_w0.05_a0.0.csv
/Users/bitsaem/Desktop/stress_project/8_6/submit_v20_rank3_tree_heavy_ordinal_stratified_k1_nearest_tq0.532_w0.00_a0.3.csv


## 확인할 결과

1. `encoding_cv_df`
   - OneHot/Ordinal, KFold/Stratified 중 어떤 조합이 좋은지
2. `pair_df`
   - k와 aggregation 최적값
3. `search_df`
   - Soft override가 Hard override보다 좋은지
4. v10 최고 `0.147603`보다 낮아졌는지


In [14]:
# v22 ExtraTrees parameter search
EXTRA_TREE_PARAM_GRID = [
    ("squared_error",1.00,1),
    ("squared_error",0.95,1),
    ("squared_error",0.90,1),
    ("absolute_error",1.00,1),
    ("absolute_error",0.95,1),
    ("absolute_error",0.90,1),
    ("absolute_error",0.95,2),
    ("absolute_error",0.90,2),
]
print("v22 parameter grid")
import pandas as pd
display(pd.DataFrame(EXTRA_TREE_PARAM_GRID,columns=["criterion","max_features","min_samples_leaf"]))
print("Replace model creation with:")
print('''model = ExtraTreesRegressor(
    n_estimators=N_ESTIMATORS_PER_SEED,
    criterion=criterion,
    max_features=max_features,
    min_samples_leaf=min_leaf,
    random_state=seed,
    n_jobs=-1,
)''')


v22 parameter grid


,criterion,max_features,min_samples_leaf
0,squared_error,1.00,1
1,squared_error,0.95,1
2,squared_error,0.90,1
3,absolute_error,1.00,1
4,absolute_error,0.95,1
5,absolute_error,0.90,1
6,absolute_error,0.95,2
7,absolute_error,0.90,2


Replace model creation with:
model = ExtraTreesRegressor(
    n_estimators=N_ESTIMATORS_PER_SEED,
    criterion=criterion,
    max_features=max_features,
    min_samples_leaf=min_leaf,
    random_state=seed,
    n_jobs=-1,
)
